# 🚀 ENZO on Google Colab — zero install

Run the whole ENZO workspace on Google's hardware — no Docker, no local
setup, reachable from any device. **Run all cells** (`Runtime → Run all`,
or `Ctrl/⌘ + F9`) and take the URL the last cell prints.

| Cell | What it does | Time |
|---|---|---|
| 1 · Install | clones this repo, installs deps, builds the UI | ~4–5 min first run, near-instant on a re-run |
| 2 · Start | boots the server in the background (UI + API on one origin) | ~10–20 s |
| 3 · Your URL | prints a Colab link (this browser) + a Cloudflare tunnel link (phone / any device) | ~1 min |
| 4 · Keep-alive | arms the disconnect prevention — see below | instant |

**Stays up for the whole session.** Two mechanisms keep ENZO running for
6+ hours: a keep-alive that resets Colab's ~90-minute idle timer every
60 s (works while this tab stays open), and a watchdog thread that pings
the server every 5 min and restarts it if it ever dies. Free Colab caps a
session at ~12 h, so a 6-hour run fits comfortably inside it. When the
session ends, one click on **Run all** brings it back (the tunnel URL
changes each time).

> The model keys are still **yours**: paste any provider key in the app
> after boot, exactly like self-hosting. Nothing is stored on our side —
> when the Colab session ends, everything on the VM is gone.


In [ ]:
# 1 · Clone + install (~4–5 min the first time; a re-run reuses what's here)
import os
import subprocess

if not os.path.exists('/content/enzo/package.json'):
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/theguysudo/ENZO.git', '/content/enzo'],
        check=True,
    )
else:
    print('ENZO already cloned — reusing this copy (server state intact).')

# Backend deps: tsx + typescript are devDependencies — --include=dev is
# required or the runtime itself never installs.
if not os.path.exists('/content/enzo/node_modules/.bin/tsx'):
    subprocess.run(
        ['npm', 'ci', '--include=dev', '--no-audit', '--no-fund'],
        cwd='/content/enzo', check=True,
    )
else:
    print('Backend dependencies already installed — reusing.')

# Frontend build. VITE_GOOGLE_AUTH=0 matches the self-hosted docker build:
# it dead-code-eliminates the Google sign-in branch (login is your keys).
if not os.path.exists('/content/enzo/synthetic-nature/dist/index.html'):
    subprocess.run(
        'VITE_GOOGLE_AUTH=0 npm ci --no-audit --no-fund && VITE_GOOGLE_AUTH=0 npm run build',
        cwd='/content/enzo/synthetic-nature', shell=True, check=True,
    )
else:
    print('Frontend already built — reusing.')

print('\n✓ ENZO is installed. Run the next cell to start the server.')


In [ ]:
# 2 · Start ENZO in the background (UI + API on one origin, port 5001)
import os
import subprocess
import time
import urllib.request

os.makedirs('/content/enzo/data', exist_ok=True)
os.makedirs('/content/enzo/generated-projects', exist_ok=True)
os.makedirs('/content/enzo/src/skills/skills', exist_ok=True)
if not os.path.exists('/content/enzo/data/memory-store.json'):
    with open('/content/enzo/data/memory-store.json', 'w') as f:
        f.write('{ "entries": [] }')
# The backend reads its memory store through this path — route it into
# data/ like the docker image does (named volume can't mount one file).
if not os.path.exists('/content/enzo/src/core/memory-store.json'):
    os.symlink('/content/enzo/data/memory-store.json', '/content/enzo/src/core/memory-store.json')

def enzo_healthy(timeout=5):
    try:
        return urllib.request.urlopen('http://127.0.0.1:5001/api/health', timeout=timeout).status == 200
    except Exception:
        return False

def start_enzo():
    log = open('/content/enzo-server.log', 'a')
    env = dict(os.environ, NODE_ENV='production', ENZO_SELF_HOSTED='1',
               ENZO_DATA_DIR='/content/enzo/data', PORT='5001')
    return subprocess.Popen(['/content/enzo/node_modules/.bin/tsx', 'index.ts'],
                            cwd='/content/enzo', stdout=log,
                            stderr=subprocess.STDOUT, env=env)

if enzo_healthy():
    print('ENZO is already running — reusing it.')
else:
    start_enzo()
    for _ in range(60):
        time.sleep(2)
        if enzo_healthy():
            break
    print('✓ Backend ready — UI + API on port 5001.' if enzo_healthy()
          else '✗ Server did not come up — open /content/enzo-server.log to see why.')


In [ ]:
# 3 · Your ENZO URL — the Colab link works in this browser; the Cloudflare
#     tunnel link works from ANY device (phone, laptop) while the session lives.
import os
import re
import subprocess
import time

try:
    from google.colab.output import eval_js
    colab_url = eval_js('google.colab.kernel.proxyPort(5001)')
except ImportError:
    colab_url = None
    print('Not in Colab — ENZO is on http://localhost:5001')

if colab_url:
    print('🔗 ENZO (this browser):', colab_url)

# Portable URL via a Cloudflare quick tunnel — free, no account. If the
# service is rate-limited or unreachable, the Colab link above still works.
if not os.path.exists('/content/cloudflared'):
    subprocess.run(
        'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared',
        shell=True,
    )

tunnel_url = None
try:
    log = open('/content/cloudflared.log', 'w')
    subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:5001'],
                     stdout=log, stderr=subprocess.STDOUT)
    for _ in range(45):
        time.sleep(2)
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cloudflared.log').read())
        if m:
            tunnel_url = m.group(0)
            break
except Exception:
    pass

if tunnel_url:
    print('🔗 ENZO (any device):', tunnel_url)
elif colab_url:
    print('Cloudflare tunnel unreachable right now — the Colab link above still works.')

print('\nRun the next cell to arm the keep-alive, then leave this tab open.')


In [ ]:
# 4 · Keep-alive — Colab idles a session out after ~90 min of tab
#     inactivity. Two mechanisms keep ENZO up for 6+ hours:
#       • the JS below resets Colab's idle timer every 60 s — works while
#         this tab stays open (free Colab caps a session ~12 h, so 6 h fits);
#       • a watchdog thread pings the server every 5 min and restarts it if
#         it ever dies, so the URL keeps serving even after a crash.
import os
import subprocess
import threading
import time
import urllib.request

from IPython.display import HTML, display

def _start_enzo():
    log = open('/content/enzo-server.log', 'a')
    env = dict(os.environ, NODE_ENV='production', ENZO_SELF_HOSTED='1',
               ENZO_DATA_DIR='/content/enzo/data', PORT='5001')
    return subprocess.Popen(['/content/enzo/node_modules/.bin/tsx', 'index.ts'],
                            cwd='/content/enzo', stdout=log,
                            stderr=subprocess.STDOUT, env=env)

def _watchdog():
    while True:
        time.sleep(300)
        try:
            urllib.request.urlopen('http://127.0.0.1:5001/api/health', timeout=10)
        except Exception:
            try:
                _start_enzo()
            except Exception:
                pass

threading.Thread(target=_watchdog, daemon=True).start()

display(HTML("""
<script>
  setInterval(() => {
    const b = document.querySelector('colab-connect-button');
    if (b) b.click();
    document.title = '\u2705 ENZO running \u00b7 ' + new Date().toLocaleTimeString();
  }, 60000);
</script>
<p><b>🟢 Keep-alive armed</b> — Colab's idle timer resets every 60 s while this
tab stays open, and the watchdog restarts the server if it ever dies. Keep the
tab open and ENZO serves for the full session (free Colab caps ~12 h; a 6-hour
run fits comfortably inside it). If the session still ends, run all cells again
— one click brings it back.</p>
"""))
